# DipRadar ML v1.0 — Modelo Inicial de Produção

**Objectivo:** Treinar o modelo definitivo XGB-v2 (30 features, rho_mean=0.21, topk_pnl=0.28)  
para scoring de dips em produção. Este notebook é o ponto de partida — dados futuros  
irão complementar e re-treinar o modelo incrementalmente.

**Arquitectura:**
- `model_up` → prevê `max_return_60d` (upside potencial)
- `model_down` → prevê `max_drawdown_60d` (risco de queda)
- `score_final = pred_up / (abs(pred_down) + ε)` → rácio risco/retorno

**Validação:** Walk-Forward CV, 5 folds, TimeSeriesSplit  
**Dataset:** `dataset_v3.parquet` (~17k amostras, 2008–2026)


## 0. Instalações

In [ ]:
!pip install xgboost lightgbm scikit-learn pandas numpy scipy joblib --quiet

## 1. Imports & Config

In [ ]:
import numpy as np
import pandas as pd
import joblib
import warnings
from pathlib import Path
from scipy.stats import spearmanr, mstats
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
np.random.seed(42)

# ── Paths ──
DATA_PATH  = Path('/content/DipRadar/experiments/ml_v2/dataset_v3.parquet')
MODEL_DIR  = Path('/content/DipRadar/models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ── Targets ──
TARGET_UP   = 'max_return_60d'
TARGET_DOWN = 'max_drawdown_60d'

# ── Feature set definitivo (30 features, XGB-v2) ──
FEATURES = [
    'macro_score', 'vix', 'vix_regime', 'sp500_trend',
    'drop_pct_today', 'drawdown_52w', 'distance_from_ma50',
    'return_5d', 'return_10d', 'return_20d', 'return_1m', 'return_3m_pre',
    'zscore_20d',
    'atr_ratio', 'volatility_20d', 'beta_60d',
    'volume_spike',
    'spy_drawdown_5d', 'sector_drawdown_5d', 'sector_relative',
    'gross_margin', 'de_ratio', 'pe_vs_fair', 'analyst_upside',
    'quality_score', 'pe_attractive',
    'drop_x_drawdown', 'vol_x_drop',
    'rsi_14', 'rsi_oversold_strength',
]

print(f'Features: {len(FEATURES)}')

## 2. Carregar Dataset

In [ ]:
df = pd.read_parquet(DATA_PATH)
df['alert_date'] = pd.to_datetime(df['alert_date'])
df = df.sort_values('alert_date').reset_index(drop=True)

FEATURES = [f for f in FEATURES if f in df.columns]
missing  = [f for f in FEATURES if f not in df.columns]
if missing:
    print(f'⚠️  Features em falta (removidas): {missing}')

print(f'Dataset: {len(df):,} amostras | {df["alert_date"].min().date()} → {df["alert_date"].max().date()}')
print(f'Features activas: {len(FEATURES)}')

## 3. Funções Auxiliares

In [ ]:
def winsorize_col(arr, limits=(0.01, 0.01)):
    return mstats.winsorize(arr, limits=limits).data

def log1p_signed(x):
    return np.sign(x) * np.log1p(np.abs(x))

def inv_log1p_signed(x):
    return np.sign(x) * (np.expm1(np.abs(x)))

def transform_targets(y_up, y_down):
    return log1p_signed(y_up), log1p_signed(y_down)

def inverse_transform(y_up_t, y_down_t):
    return inv_log1p_signed(y_up_t), inv_log1p_signed(y_down_t)

def compute_score(pred_up, pred_down, eps=0.01):
    return pred_up / (np.abs(pred_down) + eps)

def spearman_metrics(pred_up, y_up, pred_down, y_down):
    rho_up,   _ = spearmanr(pred_up,   y_up)
    rho_down, _ = spearmanr(pred_down, y_down)
    return {
        'rho_up':   round(rho_up,   4),
        'rho_down': round(rho_down, 4),
        'rho_mean': round((rho_up + rho_down) / 2, 4),
    }

def topk_pnl(pred_up, pred_down, y_up, k=0.20):
    scores = compute_score(pred_up, pred_down)
    n = max(1, int(len(scores) * k))
    idx = np.argsort(scores)[-n:]
    return round(float(np.mean(y_up[idx])), 4)

def bottomk_pnl(pred_up, pred_down, y_up, k=0.20):
    scores = compute_score(pred_up, pred_down)
    n = max(1, int(len(scores) * k))
    idx = np.argsort(scores)[:n]
    return round(float(np.mean(y_up[idx])), 4)

print('Funções definidas.')

## 4. Walk-Forward CV (Validação)

In [ ]:
N_FOLDS   = 5
TEST_SIZE = int(len(df) * 0.12)
tscv = TimeSeriesSplit(n_splits=N_FOLDS, test_size=TEST_SIZE)

def make_model_up():
    return XGBRegressor(
        n_estimators=400, learning_rate=0.05, max_depth=5,
        subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1,
        reg_lambda=1.0, random_state=42, n_jobs=-1, verbosity=0
    )

def make_model_down():
    return XGBRegressor(
        n_estimators=400, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8, reg_alpha=0.2,
        reg_lambda=2.0, random_state=42, n_jobs=-1, verbosity=0
    )

cv_results = []
print('Walk-Forward CV:')

for k, (tr_idx, te_idx) in enumerate(tscv.split(df)):
    df_tr = df.iloc[tr_idx]
    df_te = df.iloc[te_idx]
    X_tr  = df_tr[FEATURES].fillna(0).values.astype(np.float32)
    X_te  = df_te[FEATURES].fillna(0).values.astype(np.float32)
    y_up_tr   = winsorize_col(df_tr[TARGET_UP].fillna(0).values)
    y_down_tr = winsorize_col(df_tr[TARGET_DOWN].fillna(0).values)
    y_up_tr_t, y_down_tr_t = transform_targets(y_up_tr, y_down_tr)
    y_up_te   = df_te[TARGET_UP].fillna(0).values
    y_down_te = df_te[TARGET_DOWN].fillna(0).values
    m_up   = make_model_up();   m_up.fit(X_tr,   y_up_tr_t)
    m_down = make_model_down(); m_down.fit(X_tr, y_down_tr_t)
    pred_up, pred_down = inverse_transform(m_up.predict(X_te), m_down.predict(X_te))
    metrics  = spearman_metrics(pred_up, y_up_te, pred_down, y_down_te)
    pnl_top  = topk_pnl(pred_up, pred_down, y_up_te, k=0.20)
    pnl_bot  = bottomk_pnl(pred_up, pred_down, y_up_te, k=0.20)
    te_dates = df['alert_date'].iloc[te_idx]
    cv_results.append({
        'fold': k+1, 'n_test': len(te_idx),
        'date_start': te_dates.min().date(), 'date_end': te_dates.max().date(),
        **metrics, 'topk_pnl': pnl_top, 'bottomk_pnl': pnl_bot,
    })
    print(f'  Fold {k+1} | rho_up={metrics["rho_up"]:.4f} | rho_down={metrics["rho_down"]:.4f} | '
          f'rho_mean={metrics["rho_mean"]:.4f} | top20%={pnl_top:.4f} | bot20%={pnl_bot:.4f}')

df_cv = pd.DataFrame(cv_results)
print('\n=== Média CV ===')
print(df_cv[['rho_up','rho_down','rho_mean','topk_pnl','bottomk_pnl']].mean().round(4).to_string())

## 5. Treino Final (Todo o Dataset)

In [ ]:
X_all      = df[FEATURES].fillna(0).values.astype(np.float32)
y_up_all   = winsorize_col(df[TARGET_UP].fillna(0).values)
y_down_all = winsorize_col(df[TARGET_DOWN].fillna(0).values)
y_up_t, y_down_t = transform_targets(y_up_all, y_down_all)

final_model_up   = make_model_up()
final_model_down = make_model_down()
print('A treinar model_up...')
final_model_up.fit(X_all, y_up_t)
print('A treinar model_down...')
final_model_down.fit(X_all, y_down_t)
print('✅ Treino final concluído.')

## 6. Guardar Modelos & Metadata

In [ ]:
import json
from datetime import datetime

joblib.dump(final_model_up,   MODEL_DIR / 'model_up_v1.joblib')
joblib.dump(final_model_down, MODEL_DIR / 'model_down_v1.joblib')

with open(MODEL_DIR / 'features_v1.json', 'w') as f:
    json.dump(FEATURES, f, indent=2)

meta = {
    'version':    'v1.0',
    'trained_on': datetime.now().isoformat(),
    'n_samples':  len(df),
    'date_range': [str(df['alert_date'].min().date()), str(df['alert_date'].max().date())],
    'n_features': len(FEATURES),
    'features':   FEATURES,
    'targets':    {'up': TARGET_UP, 'down': TARGET_DOWN},
    'cv_mean': {
        'rho_up':      round(df_cv['rho_up'].mean(), 4),
        'rho_down':    round(df_cv['rho_down'].mean(), 4),
        'rho_mean':    round(df_cv['rho_mean'].mean(), 4),
        'topk_pnl':    round(df_cv['topk_pnl'].mean(), 4),
        'bottomk_pnl': round(df_cv['bottomk_pnl'].mean(), 4),
    },
    'notes': 'XGB-v2, 30 features, walk-forward 5 folds. Re-treinar trimestralmente.'
}

with open(MODEL_DIR / 'metadata_v1.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('Modelos guardados em:', MODEL_DIR)
for p in sorted(MODEL_DIR.iterdir()):
    print(f'  {p.name}  ({p.stat().st_size/1024:.1f} KB)')

## 7. Feature Importance

In [ ]:
import matplotlib.pyplot as plt

fi_up = pd.Series(
    final_model_up.feature_importances_, index=FEATURES
).sort_values(ascending=True).tail(20)

fig, ax = plt.subplots(figsize=(8, 7))
fi_up.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Top-20 Feature Importance — model_up (XGB v1.0)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig(MODEL_DIR / 'feature_importance_v1.png', dpi=120)
plt.show()

## 8. Exemplo de Inferência (Produção)

In [ ]:
import json

def score_alert(alert_features: dict) -> dict:
    m_up   = joblib.load(MODEL_DIR / 'model_up_v1.joblib')
    m_down = joblib.load(MODEL_DIR / 'model_down_v1.joblib')
    feats  = json.load(open(MODEL_DIR / 'features_v1.json'))
    x = np.array([[alert_features.get(f, 0.0) for f in feats]], dtype=np.float32)
    pred_up   = inv_log1p_signed(m_up.predict(x)[0])
    pred_down = inv_log1p_signed(m_down.predict(x)[0])
    score     = float(compute_score(np.array([pred_up]), np.array([pred_down]))[0])
    return {
        'pred_up':   round(pred_up, 4),
        'pred_down': round(pred_down, 4),
        'score':     round(score, 4),
        'signal':    'BUY' if score > 1.5 else ('AVOID' if score < 0.5 else 'NEUTRAL'),
    }

dummy = {f: 0.0 for f in FEATURES}
dummy.update({'atr_ratio': 0.05, 'vix': 28, 'drawdown_52w': -0.35,
              'drop_pct_today': -0.08, 'drop_x_drawdown': 0.028})
print('Exemplo de scoring:', score_alert(dummy))

## 9. Próximos Passos

### Re-treino incremental
- Correr `build_dataset.py` trimestralmente para acrescentar novos alertas
- Re-treinar com todo o histórico acumulado (expanding window)
- Comparar `rho_mean` e `topk_pnl` vs v1.0 antes de promover para produção

### Melhorias prioritárias
1. **Fix `sp500_trend`** — feature com NaN constante
2. **Melhorar `model_down`** — rho_down ≈ 0, investigar distribuição do target
3. **Backtest com TP/SL** — `score > 1.5` → entrada; SL=-15%, TP=+30%
4. **Regime split** — bull vs bear, high-VIX vs low-VIX

### Thresholds de produção (a calibrar com backtest)
```
score > 2.0  → STRONG BUY
score > 1.5  → BUY
score < 0.5  → AVOID
```
